# Bootstrap Robustness Check — AA CPD

**Purpose:** Same method as smoking_status — 100 bootstrap resamples, PC algorithm
rerun on each, edges ≥70% considered robust. Given CPD's exceptional performance
on every other check so far (λ=0.99, perfect alpha-sweep stability), this is the
final test of whether that robustness extends to resampling too.

**Input:** `pc_input_cpd_final.npy`, `pc_col_names_cpd_final.json`.

In [1]:
import numpy as np
import json
import os
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
from collections import defaultdict
import time

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
X_pc_full_cpd = np.load(os.path.join(out_dir, "pc_input_cpd_final.npy"))
with open(os.path.join(out_dir, "pc_col_names_cpd_final.json")) as f:
    col_names_cpd = json.load(f)

n_nodes = len(col_names_cpd)
outcome_idx = col_names_cpd.index("CPD")
n_samples = X_pc_full_cpd.shape[0]

def get_direct_parents_cpd(X, alpha_val=0.001):
    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in col_names_cpd]
    for i in range(n_nodes - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    cg = pc(data=X, alpha=alpha_val, indep_test=fisherz, stable=True,
            uc_rule=0, uc_priority=2, background_knowledge=bk,
            verbose=False, show_progress=False, node_names=col_names_cpd)

    adj = cg.G.graph
    direct_parents = set()
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            if adj[i,j] == -1 and adj[j,i] == 1 and col_names_cpd[j] == "CPD":
                direct_parents.add(col_names_cpd[i])
            elif adj[i,j] == 1 and adj[j,i] == -1 and col_names_cpd[i] == "CPD":
                direct_parents.add(col_names_cpd[j])
            elif adj[i,j] == -1 and adj[j,i] == -1:
                if col_names_cpd[i] == "CPD":
                    direct_parents.add(col_names_cpd[j])
                elif col_names_cpd[j] == "CPD":
                    direct_parents.add(col_names_cpd[i])
    return direct_parents

n_bootstraps = 100
edge_counts_cpd = defaultdict(int)

np.random.seed(0)
start = time.time()
for b in range(n_bootstraps):
    boot_idx = np.random.choice(n_samples, n_samples, replace=True)
    X_boot = X_pc_full_cpd[boot_idx]
    parents = get_direct_parents_cpd(X_boot)
    for gene in parents:
        edge_counts_cpd[gene] += 1
    if (b + 1) % 20 == 0:
        print(f"Completed {b+1}/{n_bootstraps}, elapsed {time.time()-start:.1f}s")

print(f"\nTotal time: {time.time()-start:.1f}s")
print("\nEdge stability across 100 bootstraps:")
for gene, count in sorted(edge_counts_cpd.items(), key=lambda x: -x[1]):
    print(f"  {gene}: {count}/100 ({count}%)")

c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Completed 20/100, elapsed 0.1s
Completed 40/100, elapsed 0.1s
Completed 60/100, elapsed 0.2s
Completed 80/100, elapsed 0.2s
Completed 100/100, elapsed 0.3s

Total time: 0.3s

Edge stability across 100 bootstraps:
  exm2277017-0_T_R_1989215336: 100/100 (100%)
  exm2265806-0_B_R_1984855512: 91/100 (91%)
  exm318398-0_B_F_1922126681: 74/100 (74%)
